## Write a solution to find the number of times each student attended each exam.

Return the result table ordered by student_id and subject_name.

The result format is in the following example.

 

Example 1:

Input: 
Students table:
+------------+--------------+
| student_id | student_name |
+------------+--------------+
| 1          | Alice        |
| 2          | Bob          |
| 13         | John         |
| 6          | Alex         |
+------------+--------------+
Subjects table:
+--------------+
| subject_name |
+--------------+
| Math         |
| Physics      |
| Programming  |
+--------------+
Examinations table:
+------------+--------------+
| student_id | subject_name |
+------------+--------------+
| 1          | Math         |
| 1          | Physics      |
| 1          | Programming  |
| 2          | Programming  |
| 1          | Physics      |
| 1          | Math         |
| 13         | Math         |
| 13         | Programming  |
| 13         | Physics      |
| 2          | Math         |
| 1          | Math         |
+------------+--------------+
Output: 
+------------+--------------+--------------+----------------+
| student_id | student_name | subject_name | attended_exams |
+------------+--------------+--------------+----------------+
| 1          | Alice        | Math         | 3              |
| 1          | Alice        | Physics      | 2              |
| 1          | Alice        | Programming  | 1              |
| 2          | Bob          | Math         | 1              |
| 2          | Bob          | Physics      | 0              |
| 2          | Bob          | Programming  | 1              |
| 6          | Alex         | Math         | 0              |
| 6          | Alex         | Physics      | 0              |
| 6          | Alex         | Programming  | 0              |
| 13         | John         | Math         | 1              |
| 13         | John         | Physics      | 1              |
| 13         | John         | Programming  | 1              |
+------------+--------------+--------------+----------------+

## taking pandas dataframe from leet code and than convert it in Spark data frame

In [1]:
import pandas as pd
data = [[1, 'Alice'], [2, 'Bob'], [13, 'John'], [6, 'Alex']]
students = pd.DataFrame(data, columns=['student_id', 'student_name']).astype({'student_id':'Int64', 'student_name':'object'})
data = [['Math'], ['Physics'], ['Programming']]
subjects = pd.DataFrame(data, columns=['subject_name']).astype({'subject_name':'object'})
data = [[1, 'Math'], [1, 'Physics'], [1, 'Programming'], [2, 'Programming'], [1, 'Physics'], [1, 'Math'], [13, 'Math'], [13, 'Programming'], [13, 'Physics'], [2, 'Math'], [1, 'Math']]
examinations = pd.DataFrame(data, columns=['student_id', 'subject_name']).astype({'student_id':'Int64', 'subject_name':'object'})

In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Student and Examination").getOrCreate()
student_df=spark.createDataFrame(students)
subject_df=spark.createDataFrame(subjects)
examination_df=spark.createDataFrame(examinations)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 17:22:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/07/06 17:23:14 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [14]:
stu_subDf=student_df.crossJoin(subject_df)
stu_subDf.show()

+----------+------------+------------+
|student_id|student_name|subject_name|
+----------+------------+------------+
|         1|       Alice|        Math|
|         1|       Alice|     Physics|
|         1|       Alice| Programming|
|         2|         Bob|        Math|
|         2|         Bob|     Physics|
|         2|         Bob| Programming|
|        13|        John|        Math|
|        13|        John|     Physics|
|        13|        John| Programming|
|         6|        Alex|        Math|
|         6|        Alex|     Physics|
|         6|        Alex| Programming|
+----------+------------+------------+



In [4]:
examination_df.show()

+----------+------------+
|student_id|subject_name|
+----------+------------+
|         1|        Math|
|         1|     Physics|
|         1| Programming|
|         2| Programming|
|         1|     Physics|
|         1|        Math|
|        13|        Math|
|        13| Programming|
|        13|     Physics|
|         2|        Math|
|         1|        Math|
+----------+------------+



In [11]:
from pyspark.sql.functions import col,count
df=examination_df.groupBy("student_id","subject_name").agg(count("*").alias("attended_exams"))
df.show()

+----------+------------+--------------+
|student_id|subject_name|attended_exams|
+----------+------------+--------------+
|         1|        Math|             3|
|         1|     Physics|             2|
|         1| Programming|             1|
|         2| Programming|             1|
|        13|        Math|             1|
|        13| Programming|             1|
|        13|     Physics|             1|
|         2|        Math|             1|
+----------+------------+--------------+



In [ ]:
final_df=stu_subDf.join(df,["student_id","subject_name"], "left").fillna({"attended_exams":0}).orderBy("student_id","subject_name")
final_df.show()

+----------+------------+------------+--------------+
|student_id|subject_name|student_name|attended_exams|
+----------+------------+------------+--------------+
|         1|        Math|       Alice|             3|
|         1|     Physics|       Alice|             2|
|         1| Programming|       Alice|             1|
|         2|        Math|         Bob|             1|
|         2|     Physics|         Bob|             0|
|         2| Programming|         Bob|             1|
|         6|        Math|        Alex|             0|
|         6|     Physics|        Alex|             0|
|         6| Programming|        Alex|             0|
|        13|        Math|        John|             1|
|        13|     Physics|        John|             1|
|        13| Programming|        John|             1|
+----------+------------+------------+--------------+



26/07/08 05:26:16 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 3311064 ms exceeds timeout 120000 ms
26/07/08 05:26:17 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/08 05:26:17 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$